# HAM10000 exploratory analysis

Use the [project Drive](https://drive.google.com/drive/folders/1nf-klvmQ2LyqpmI9ZU4jXEwWcavUjNlt) files below. Set `DATA_DIR` to the folder containing them before starting the notebook kernel.

- Keep `ISIC2018_Task3_Training_Input.zip` as an archive.
- Extract `ISIC2018_Task3_Training_GroundTruth.zip` and `ISIC2018_Task3_Test_GroundTruth.zip` into that folder, retaining their internal directories.
- Keep `ISIC2018_Task3_Training_LesionGroupings.csv` alongside the archives.

The test-label CSV is used for class counts only.

In [ ]:
import os
import sys
from collections import Counter
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from src.data import CLASSES, load_metadata, load_split, load_test, resolve_data_dir
from src.download_data import TRAIN_IMAGES_ZIP

folder = resolve_data_dir()
figures = root / "results/figures"
figures.mkdir(parents=True, exist_ok=True)
train = load_metadata(folder)
assert train.notna().all().all()

## Class balance
Compare the full challenge training release with the HAM10000 row of [Tschandl et al. (2018), Table 1](https://arxiv.org/pdf/1803.10417#page=5). The test column uses only the official ground-truth CSV.

In [ ]:
paper = pd.Series([327, 514, 1099, 115, 1113, 6705, 142], index=CLASSES)
counts = pd.DataFrame({"Training images": train.label.value_counts(), "Table 1": paper,
                       "Test labels": load_test(folder).label.value_counts()}).loc[CLASSES]
display(counts)
assert counts["Training images"].eq(paper).all()
ax = counts["Training images"].plot.barh(color="#326678", figsize=(7, 4))
ax.bar_label(ax.containers[0], padding=3)
ax.set(xlabel="Images", ylabel="Diagnosis", title="Training images by diagnosis", xlim=(0, 7600))
ax.figure.tight_layout()
ax.figure.savefig(figures / "eda_class_counts.png", dpi=150)
plt.show()

## Repeated lesions and diagnosis confirmation

In [ ]:
assert train.groupby("lesion_id").label.nunique().eq(1).all()
sizes = train.groupby(["label", "lesion_id"]).size().rename("images")
by_class = sizes.groupby(level="label")
summary = by_class.agg(["count", "sum", "mean", "max"])
summary.columns = ["Lesions", "Images", "Images per lesion", "Maximum"]
summary["Multi-image lesions"] = sizes.gt(1).groupby(level="label").sum()
display(summary.loc[CLASSES].round(2))
display(sizes.value_counts().sort_index().rename_axis("Images per lesion").rename("Lesions"))
confirmation = pd.crosstab(train.label, train.diagnosis_confirm_type).loc[CLASSES]
display(confirmation)

## Image dimensions and artifact review
Check the dimensions of every training-release JPEG. For the grid, sample eight images per diagnosis from the committed **training fold**, using seed 0. Equal row sizes make the classes easy to inspect but cannot estimate dataset-wide artifact prevalence.

In [ ]:
eligible = train.merge(load_split(), on="image_id", validate="one_to_one")
eligible = eligible.loc[eligible.split == "train"].sort_values("image_id")
sample = eligible.groupby("label").sample(n=8, random_state=0)
wanted = set(sample.image_id)
thumbnails = {}
dimensions = Counter()
archive_source = folder / TRAIN_IMAGES_ZIP
if os.environ.get("TRAIN_IMAGE_ZIP_URL"):
    # Optional direct download for a runtime without a mounted Drive.
    with urlopen(os.environ["TRAIN_IMAGE_ZIP_URL"]) as response:
        archive_source = BytesIO(response.read())
with ZipFile(archive_source) as archive:
    members = [p for p in archive.namelist() if p.lower().endswith(".jpg") and not p.startswith("__MACOSX/")]
    assert len(members) == len(train) and {Path(p).stem for p in members} == set(train.image_id)
    for member in members:
        image_id = Path(member).stem
        with archive.open(member) as stream, Image.open(stream) as image:
            dimensions[image.size] += 1
            if image_id in wanted:
                image.thumbnail((250, 200))
                thumbnails[image_id] = image.convert("RGB")
del archive_source
print("Training JPEG sizes (width, height):", dict(dimensions))

In [ ]:
fig, rows = plt.subplots(len(CLASSES), 8, figsize=(16, 12), layout="constrained")
for label, axes in zip(CLASSES, rows):
    for ax, image_id in zip(axes, sample.loc[sample.label == label, "image_id"]):
        ax.imshow(thumbnails[image_id])
        ax.set_title(image_id, fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0].set_ylabel(label, fontsize=12)
fig.suptitle("Eight randomly sampled training images per diagnosis")
fig.savefig(figures / "eda_training_grid.png", dpi=150)
plt.show()

## Findings

EDA used the project Drive archives. Counts cover the 10,015-image challenge training release, and the visual grid uses eight randomly selected images per class from the committed training fold with seed 0.

| Class | Training images | Test labels |
|---|---:|---:|
| akiec | 327 | 43 |
| bcc | 514 | 93 |
| bkl | 1,099 | 217 |
| df | 115 | 44 |
| mel | 1,113 | 171 |
| nv | 6,705 | 909 |
| vasc | 142 | 35 |
| **Total** | **10,015** | **1,512** |

- All seven training counts match [Tschandl et al., Table 1](https://arxiv.org/pdf/1803.10417#page=5). Nevi account for 67.0% of training images, and dermatofibroma accounts for 1.1%.
- The 10,015 images represent 7,470 lesions. Of those, 1,956 (26.2%) have multiple images, with up to six images per lesion. Melanoma has the highest mean, at 1.81 images per lesion.
- Histopathology confirms 5,340 images (53.3%), including every melanoma, basal cell carcinoma, and akiec image. Serial imaging confirms 3,704 nevus images.
- Every training JPEG is 600 × 450 pixels.
- The 56-image grid shows visible hairs, dark circular borders, and glare. I found no recognizable ruler or clear ink marking in this sample. The class-balanced grid is a visual check with eight images per class.